# xfig_25 — SOTA Comparison Bubble Chart

Positions our results alongside SleepFounder, OSF, and SleepMaMi.

- **x-axis** = pre-training data volume (hours, log scale)
- **y-axis** = AUROC on matching tasks
- **Filled markers** = method uses EEG; **open markers** = cardio-only (no EEG)

⚠️ Different evaluation protocols — comparison is approximate.

Idea #25 from `docs/NEW_PLOT_IDEAS.md`.

**Data**: hardcoded SOTA numbers + analysis.csv for our results.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
EXPLORE_DIR    = NSRR_TOOLS / "results" / "paper_figures" / "explore"
FINAL_OUT      = EXPLORE_DIR / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)
TABLES_DIR     = NSRR_TOOLS / "results" / "tables"

# Add explore utils to path
_nb_dir = EXPLORE_DIR / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.data_explore import (
    set_root, load_analysis, load_analysis_all_k,
    load_heatmap, load_parquets, load_modality_table,
    subject_predictions, subject_correctness_matrix, CONTEXT_TO_MIN, CTX_ORDER,
)
from utils import panels_explore as xp

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

set_root(WORKSPACE_ROOT)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 7,
})

# ── Constants ──────────────────────────────────────────────────────────────────
MAIN_TASKS = ["sex_binary", "bmi_binary", "age_class",
              "sleep_efficiency_binary", "apnea_binary"]
TASK_LABEL = xp.TASK_LABEL

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND'}")


In [ ]:
# Load our results for reference (numbers in panels_explore.SOTA_DATA are hardcoded)
df = load_analysis("phase0_v3", split="test", k="all")

# Optional: verify our hardcoded numbers against the actual CSV
for task, head, ctx in [
    ("apnea_binary", "lstm", "120m"),
    ("apnea_binary", "transformer", "120m"),
    ("sex_binary",   "lstm", "120m"),
    ("sex_binary",   "transformer", "240m"),
]:
    row = df[(df.task == task) & (df.head == head) & (df.context_length == ctx)]
    if not row.empty:
        print(f"{task}/{head}/{ctx}: {row['mean_prob_auroc'].values[0]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
xp.sota_bubble_panel(ax)
ax.set_title("SOTA comparison (⚠ different eval protocols — approximate)",
             fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
fig.savefig(str(FINAL_OUT / 'xfig_25_sota_bubble.pdf'), bbox_inches='tight')
fig.savefig(str(FINAL_OUT / 'xfig_25_sota_bubble.png'), dpi=150, bbox_inches='tight')
print('Saved →', FINAL_OUT / 'xfig_25_sota_bubble.pdf')